# Lekce 03 - Agentní návrhové vzory

V této lekci prozkoumáme tři základní návrhové vzory pro vytváření efektivních AI agentů:

1. **Jasné instrukce pro agenta** — Vytváření přesných, rolí definujících výzev, které řídí chování agenta  
2. **Strukturovaný výstup s Pydantic modely** — Zajištění, že agenti vracejí předvídatelná, validovaná data  
3. **Agenti s jednou odpovědností** — Navrhování zaměřených agentů, kteří každý dělají jednu věc dobře

Každý vzor použijeme na scénář **doporučovače cestovních destinací**, postupně vybudujeme systém, který může navrhovat destinace, kontrolovat dostupnost a řešit logistiku.


## Nastavení


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Vzor 1: Jasné instrukce pro agenta

Nejúčinnější vzor je také nejjednodušší: napsat jasné, podrobné instrukce pro vašeho agenta.

Dobré instrukce definují:
- **Kdo** je agent (persona a tón)
- **Co** by měl dělat (krok za krokem povinnosti)
- **Jak** by se měl chovat (omezení a styl)

Níže vytváříme agent cestovního concierge s explicitními instrukcemi, které formují každou odpověď, kterou produkuje.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Vzor 2: Strukturovaný výstup s Pydantic modely

Volný text je užitečný pro konverzaci, ale následující systémy potřebují strukturovaná data.  
Spojením **Pydantic modelů** s **nástrojovou funkcí** můžeme:

- Definovat přesné schéma pro výstup agenta  
- Automaticky ověřovat odpovědi  
- Spolehlivě integrovat výsledky agenta do logiky aplikace  

Také zavádíme nástroj, který vrací podrobnosti o destinaci, takže agent zakládá svá doporučení na reálných datech.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Vzor 3: Agent s jedinou odpovědností

Komplexní úkoly mají prospěch z rozdělení práce mezi více zaměřených agentů, z nichž každý má jedinou odpovědnost:

- **Expert na destinace**, který zná místa a dostupnost
- **Plánovač logistiky**, který se stará o lety, hotely a itineráře

To odráží zásadu softwarového inženýrství *oddělení starostí* — každý agent je snazší testovat, udržovat a zlepšovat samostatně.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Shrnutí

V této lekci jsme aplikovali tři agentní designové vzory na scénář cestovního doporučování:

| Vzor | Hlavní myšlenka | Výhoda |
|---|---|---|
| **Jasné pokyny** | Definujte osobnost, odpovědnosti a omezení předem | Konzistentní chování agenta v souladu se značkou |
| **Strukturovaný výstup** | Použijte modely Pydantic jako formát odpovědi | Validní, strojově čitelné výsledky |
| **Jedna odpovědnost** | Dejte každému agentovi jednu konkrétní úlohu | Snazší testování, údržba a skládání |

Tyto vzory se přirozeně skládají — můžete kombinovat jasné pokyny se strukturovaným výstupem v rámci agenta s jednou odpovědností a vytvořit tak robustní systémy připravené pro produkci.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o vyloučení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). I když se snažíme o přesnost, mějte prosím na paměti, že automatické překlady mohou obsahovat chyby nebo nepřesnosti. Původní dokument v jeho mateřském jazyce by měl být považován za autoritativní zdroj. Pro důležité informace doporučujeme profesionální lidský překlad. Nejsme odpovědní za jakékoliv nedorozumění nebo nesprávné výklady vyplývající z použití tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
